# Caos I: la ecuación de Lorenz

## **Ejercicio 1)** Importando librerías

**1)** Importe las librerías `numpy` para operar con arrays, `scipy.signal` para analizar señales de datos y `matplotlib.pyplot` para graficar.

**2)** Suba el archivo "libreria" `ode.py` que fabricó en el práctico 4), e impórtelo.

In [ ]:
# a)
import numpy as np
#import scipy as sp
import scipy.signal as sgn
#import scipy.linalg as linalg
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# b)
#import ode

def euler(f,x,t,h,p):
    """
    Calcula un paso de integración del método de Euler.

    Argumentos de entrada:

        f : R^n,R -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numérica de

        x(t+h) : R^n

    según el método de Euler.

    # Ejemplos:
    """
    return x+h*f(x,t,p)

def rk4(f,x,t,h,p):
    """
    Calcula un paso de integración del método de Runge Kutta orden 4.

    Argumentos de entrada:

        f : R^n -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numérica de

        x(t+h) : R^n

    según el método de Runge-Kutta de orden 4.

    # Ejemplos:
    """
    k1 = f(x,t,p)
    k2 = f(x+0.5*h*k1,t+0.5*h,p)
    k3 = f(x+0.5*h*k2,t+0.5*h,p)
    k4 = f(x+h*k3,t+h,p)
    return x+h*(k1+2.0*k2+2.0*k3+k4)/6.0

def integrador_ode(m,f,xa,a,b,k,p,c=lambda x,t,p:x):
    """
    Integra numéricamente la ODE

        dx/dt = f(x,t)

    sobre el intervalo t:[a,b] usando k pasos de integración y el método m, bajo condicion inicial x(a)=x0.
    No es necesario que a<b.

    Argumentos de entrada:

        m = metodo de integracion (ej. euler, rk2, etc.)
        f : R^n -> R^n
        xa = condicion inicial : R
        a = tiempo inicial : R
        b = tiempo final : R
        k = num. pasos de integracion : N
        p = parametros : R^q
        c = función condicionante : R^n,R,p -> R^n

    Retorna:

        t : R^{k+1} , t_j = a+j*h para j=0,1,...,k
        w : R^{n,k+1} , w_ij = x_i(t_j) para i=0,1,...,n-1 y j=0,1,...,k

    donde a+k*dt = b.
    """
    assert k>0
    n = len(xa)
    h = (b-a)/k
    w = np.zeros((n,k+1)) # Produce un array con forma y tipo especificada con los parametros,
                          # lleno de ceros. la forma puede ser espcificada con un entero o tupla (n,k+1)
    t = np.zeros(k+1)
    w[:,0] = xa           # actualiza la posicion inicial (columna de indice 0) de las variables con los valores
                          # de las condiciones iniciales
    t[0] = a              # actualiza la posicion cero con el valor del tiempo inicial

    for j in range(k):    #Aca se produce la iteración en j

        t[j+1] = t[j] + h                # iteracion tiempo
        w[:,j+1] = m(f,w[:,j],t[j],h,p)  # iteracion de w
        w[:,j+1] = c(w[:,j+1],t[j+1],p)  # condicion sobre w

    return t,w

## Ejercicio 2)

Considere el sistema de Ecuaciones Diferenciales Ordinarias de Lorenz

\begin{eqnarray}
\frac{dx}{dt} & = & s(y-x) \\
\frac{dy}{dt} & = & rx-y-xz \\
\frac{dz}{dt} & = & xy-bz
\end{eqnarray}

Sea $p=(s,r,b)$ el vector de parámetros.

**1)**
Usando el método de Runge-Kutta de orden 4, resuelva la ODE para las siguientes elecciones de las condiciones iniciales, de los parámetros y del intervalo de tiempo de integración:

&nbsp; &nbsp; **i)** $(x(0),y(0),z(0))=(1,0.5,0.1)$, $p=(10,0.5,8/3)$ y $t\in [0,10]$.

&nbsp; &nbsp; **ii)** $(x(0),y(0),z(0))=(1,0.5,0.1)$, $p=(10,10,8/3)$ y $t\in [0,20]$.

&nbsp; &nbsp; **iii)** $(x(0),y(0),z(0))=(1,0.5,0.1)$, $p=(10,28,8/3)$ y $t\in [0,50]$

**2)** Para cada caso calculado en el inciso **1)**, grafique, en un mismo gráfico, las curvas en función del tiempo $x(t)$ vs $t$, $y(t)$ vs $t$ y $z(t)$ vs $t$.

**3)**
Para cada caso calculado en el inciso **1)**, grafique paramétricamente en función del tiempo, y en 3 dimensiones, el vector que describe la trayectoria de la solución, i.e. grafique $(x(t),y(t),z(t))$ vs $t$.

In [ ]:
# ==============================================================================
# Ejercicio 2: Sistema de Lorenz y simulación numérica con RK4
# ==============================================================================

# Definición del campo vectorial del sistema de Lorenz
def f_lorenz(x, t, p):
    """
    Ecuaciones diferenciales del sistema de Lorenz:
        dx/dt = s * (y - x)
        dy/dt = r * x - y - x * z
        dz/dt = x * y - b * z
    
    Argumentos:
        x : np.ndarray de dimensión 3, estado [x, y, z]
        t : float, tiempo actual
        p : tuple/list de dimensión 3, parámetros (s, r, b)
    
    Retorna:
        np.ndarray de dimensión 3 con las derivadas temporales
    """
    s, r, b = p
    dxdt = s * (x[1] - x[0])
    dydt = r * x[0] - x[1] - x[0] * x[2]
    dzdt = x[0] * x[1] - b * x[2]
    return np.array([dxdt, dydt, dzdt])

# ------------------------------------------------------------------------------
# 2.1) Integración numérica con RK4 para los 3 casos
# ------------------------------------------------------------------------------
x0 = np.array([1.0, 0.5, 0.1])  # Condición inicial
h = 0.01                         # Paso temporal de integración

# Caso i: p = (10, 0.5, 8/3), t in [0, 10]
p_i = (10.0, 0.5, 8.0 / 3.0)
k_i = int(round((10.0 - 0.0) / h))
t_i, w_i = integrador_ode(rk4, f_lorenz, x0, 0.0, 10.0, k_i, p_i)

# Caso ii: p = (10, 10, 8/3), t in [0, 20]
p_ii = (10.0, 10.0, 8.0 / 3.0)
k_ii = int(round((20.0 - 0.0) / h))
t_ii, w_ii = integrador_ode(rk4, f_lorenz, x0, 0.0, 20.0, k_ii, p_ii)

# Caso iii: p = (10, 28, 8/3), t in [0, 50]
p_iii = (10.0, 28.0, 8.0 / 3.0)
k_iii = int(round((50.0 - 0.0) / h))
t_iii, w_iii = integrador_ode(rk4, f_lorenz, x0, 0.0, 50.0, k_iii, p_iii)

print("Integración completada exitosamente para los tres casos.")
print(f"Caso i:   {len(t_i)} puntos calculados en t in [0, 10]")
print(f"Caso ii:  {len(t_ii)} puntos calculados en t in [0, 20]")
print(f"Caso iii: {len(t_iii)} puntos calculados en t in [0, 50]")

# ------------------------------------------------------------------------------
# 2.2) Gráficos de x(t), y(t), z(t) vs t en un mismo gráfico para cada caso
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=False)

# Caso i
axes[0].plot(t_i, w_i[0, :], label=r'$x(t)$', color='tab:blue', lw=1.5)
axes[0].plot(t_i, w_i[1, :], label=r'$y(t)$', color='tab:orange', lw=1.5)
axes[0].plot(t_i, w_i[2, :], label=r'$z(t)$', color='tab:green', lw=1.5)
axes[0].set_title(r'Caso i: $s=10$, $r=0.5$, $b=8/3$ ($r < 1$: origen $(0,0,0)$ globalmente asintóticamente estable)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tiempo $t$', fontsize=11)
axes[0].set_ylabel('Amplitud', fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper right', framealpha=0.9)

# Caso ii
axes[1].plot(t_ii, w_ii[0, :], label=r'$x(t)$', color='tab:blue', lw=1.5)
axes[1].plot(t_ii, w_ii[1, :], label=r'$y(t)$', color='tab:orange', lw=1.5)
axes[1].plot(t_ii, w_ii[2, :], label=r'$z(t)$', color='tab:green', lw=1.5)
axes[1].set_title(r'Caso ii: $s=10$, $r=10$, $b=8/3$ ($1 < r < r_H$: punto fijo $C^+$ estable)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tiempo $t$', fontsize=11)
axes[1].set_ylabel('Amplitud', fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right', framealpha=0.9)

# Caso iii
axes[2].plot(t_iii, w_iii[0, :], label=r'$x(t)$', color='tab:blue', lw=1.0)
axes[2].plot(t_iii, w_iii[1, :], label=r'$y(t)$', color='tab:orange', lw=1.0)
axes[2].plot(t_iii, w_iii[2, :], label=r'$z(t)$', color='tab:green', lw=1.0)
axes[2].set_title(r'Caso iii: $s=10$, $r=28$, $b=8/3$ ($r > r_H \approx 24.74$: régimen caótico aperiódico)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Tiempo $t$', fontsize=11)
axes[2].set_ylabel('Amplitud', fontsize=11)
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2.3) Gráficos paramétricos 3D: (x(t), y(t), z(t)) vs t usando plot3D
# ------------------------------------------------------------------------------
fig = plt.figure(figsize=(18, 6))

# Caso i en 3D
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot3D(w_i[0, :], w_i[1, :], w_i[2, :], color='tab:blue', lw=1.5, label='Trayectoria')
ax1.scatter(w_i[0, 0], w_i[1, 0], w_i[2, 0], color='red', s=60, marker='o', label='CI (1, 0.5, 0.1)')
ax1.scatter(0, 0, 0, color='black', s=60, marker='*', label='Punto fijo (0,0,0)')
ax1.set_title(r'Caso i: $r=0.5$ (Convergencia a (0,0,0))', fontsize=12, fontweight='bold')
ax1.set_xlabel('$x$', fontsize=11)
ax1.set_ylabel('$y$', fontsize=11)
ax1.set_zlabel('$z$', fontsize=11)
ax1.view_init(elev=25, azim=45)
ax1.legend(loc='upper left', fontsize=9)

# Caso ii en 3D
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot3D(w_ii[0, :], w_ii[1, :], w_ii[2, :], color='tab:orange', lw=1.2, label='Trayectoria')
ax2.scatter(w_ii[0, 0], w_ii[1, 0], w_ii[2, 0], color='red', s=60, marker='o', label='CI (1, 0.5, 0.1)')
# Punto fijo C+ para r=10: x=sqrt(8/3*9)=sqrt(24), y=sqrt(24), z=9
c_plus = (np.sqrt(8.0 / 3.0 * 9.0), np.sqrt(8.0 / 3.0 * 9.0), 9.0)
ax2.scatter(c_plus[0], c_plus[1], c_plus[2], color='black', s=60, marker='*', label=r'Punto fijo $C^+$')
ax2.set_title(r'Caso ii: $r=10$ (Espiral hacia $C^+$)', fontsize=12, fontweight='bold')
ax2.set_xlabel('$x$', fontsize=11)
ax2.set_ylabel('$y$', fontsize=11)
ax2.set_zlabel('$z$', fontsize=11)
ax2.view_init(elev=25, azim=45)
ax2.legend(loc='upper left', fontsize=9)

# Caso iii en 3D
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
ax3.plot3D(w_iii[0, :], w_iii[1, :], w_iii[2, :], color='purple', lw=0.6, alpha=0.85, label='Atractor extraño')
ax3.scatter(w_iii[0, 0], w_iii[1, 0], w_iii[2, 0], color='red', s=60, marker='o', label='CI (1, 0.5, 0.1)')
ax3.set_title(r'Caso iii: $r=28$ (Atractor Caótico de Lorenz)', fontsize=12, fontweight='bold')
ax3.set_xlabel('$x$', fontsize=11)
ax3.set_ylabel('$y$', fontsize=11)
ax3.set_zlabel('$z$', fontsize=11)
ax3.view_init(elev=25, azim=45)
ax3.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

## Ejercicio 3)

**1)** Repita el caso **1.iii)** del ejercicio **2)**, para $t\in [0,300]$, y eliminando el transiente que se observa antes de "converger" al atractor.

**2)** Identifique los máximos locales de la curva $z(t)$ obtenida en el inciso **1)**, y enumérelos como $z_1,z_2,...,z_i,...,z_n$.

**3)** Grafique $z_{i+1}$ vs $z_i$ para todo $i\in\{1,2,...,n-1\}$.

In [ ]:
# ==============================================================================
# Ejercicio 3: Análisis del atractor y Mapa de Retorno de Lorenz
# ==============================================================================

# ------------------------------------------------------------------------------
# 3.1) Simulación prolongada para t in [0, 300] y eliminación del transiente inicial
# ------------------------------------------------------------------------------
p_lorenz = (10.0, 28.0, 8.0 / 3.0)
x0 = np.array([1.0, 0.5, 0.1])
h = 0.01
k_largo = int(round((300.0 - 0.0) / h))

t_long, w_long = integrador_ode(rk4, f_lorenz, x0, 0.0, 300.0, k_largo, p_lorenz)

# Eliminamos el transiente inicial (t < 50)
t_transiente = 50.0
mask_atractor = t_long >= t_transiente

t_atr = t_long[mask_atractor]
x_atr = w_long[0, mask_atractor]
y_atr = w_long[1, mask_atractor]
z_atr = w_long[2, mask_atractor]

print(f"Puntos totales integrados: {len(t_long)}")
print(f"Puntos sobre el atractor (t >= {t_transiente}): {len(t_atr)}")

# ------------------------------------------------------------------------------
# 3.2) Identificación de máximos locales de z(t)
# ------------------------------------------------------------------------------
# Usamos scipy.signal.find_peaks sobre z_atr.
# Fijamos distance=30 pasos (0.3 unidades de tiempo) para evitar duplicados locales.
peaks_indices, _ = sgn.find_peaks(z_atr, distance=30)

t_peaks = t_atr[peaks_indices]
z_peaks = z_atr[peaks_indices]
n_peaks = len(z_peaks)

print(f"Se identificaron n = {n_peaks} máximos locales en z(t): z_1, z_2, ..., z_{n_peaks}")
print(f"Primeros 5 máximos locales: {np.round(z_peaks[:5], 3)}")

# Gráfico de z(t) y los máximos detectados (zoom en t in [50, 90] para apreciar el detalle)
plt.figure(figsize=(14, 5))
mask_zoom = (t_atr >= 50.0) & (t_atr <= 90.0)
mask_peaks_zoom = (t_peaks >= 50.0) & (t_peaks <= 90.0)

plt.plot(t_atr[mask_zoom], z_atr[mask_zoom], color='tab:green', lw=1.2, label=r'$z(t)$ (trayectoria sobre el atractor)')
plt.plot(t_peaks[mask_peaks_zoom], z_peaks[mask_peaks_zoom], 'ro', markersize=6, label=r'Máximos locales $z_i$')
plt.title(r'3.2) Serie temporal $z(t)$ y máximos locales detectados (detalle en $t \in [50, 90]$)', fontsize=13, fontweight='bold')
plt.xlabel('Tiempo $t$', fontsize=11)
plt.ylabel('$z(t)$', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

# ------------------------------------------------------------------------------
# 3.3) Graficar z_{i+1} vs z_i (Mapa de Lorenz)
# ------------------------------------------------------------------------------
z_i = z_peaks[:-1]
z_i_plus_1 = z_peaks[1:]

z_min, z_max = np.min(z_peaks) - 1.0, np.max(z_peaks) + 1.0

plt.figure(figsize=(8, 7))
plt.plot(z_i, z_i_plus_1, 'o', color='crimson', markersize=3.5, alpha=0.8, label=r'$(z_i, z_{i+1})$')
plt.plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.6, label=r'Identidad $z_{i+1} = z_i$')
plt.title('3.3) Mapa de Retorno de Lorenz ($z_{i+1}$ vs $z_i$)', fontsize=14, fontweight='bold')
plt.xlabel(r'$z_i$ (máximo local $i$)', fontsize=12)
plt.ylabel(r'$z_{i+1}$ (máximo local $i+1$)', fontsize=12)
plt.xlim(z_min, z_max)
plt.ylim(z_min, z_max)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

## Ejercicio 4)

**1)** Con la intención de imitar un set de datos, agregue ruido Gaussiano de varianza $\sigma^2=1$ a la curva $z(t)$ vs $t$ generada en el inciso **1)** del ejercicio **3)**. Llamaremos a esta nueva curva $\tilde{z}(t)$ vs $t$. Grafíquela.

**2)** Utilice algún algoritmo de suavizado de curvas (ver función `savgol_filter` de la librería `scipy.signal`) para suavizar $\tilde{z}(t)$ vs $t$. Llame $\hat{z}(t)$ vs $t$ a la curva suavizada.

**3)** Utilice la curva suavizada del inciso **2)** para identificar los tiempos $\hat{t}_1$,$\hat{t}_2$,...,$\hat{t}_n$ en los cuales ocurre los máximos locales $\hat{z}_1=\hat{z}(\hat{t}_1)$, $\hat{z}_2=\hat{z}(\hat{t}_2)$, ...,$\hat{z}_n=\hat{z}(\hat{t}_n)$ correspondientes.

Calcule además los valores de la curva sin suavizar en dichos tiempos, i.e. calcule $\tilde{z}_1=\tilde{z}(\hat{t_1})$, $\tilde{z}_2=\tilde{z}(\hat{t_2})$, ..., $\tilde{z}_n=\tilde{z}(\hat{t_n})$.

**4)** Grafique la curva suavizada $\tilde{z}(t)$ vs $t$.

Agregue al gráfico los máximos $(\tilde{z}_i,\hat{t}_i)$ con puntos azules, y los máximos $(\hat{z}_i,\hat{t}_i)$ con puntos rojos.

Que observa?

**5)** Grafique $\tilde{z}_{i+1}$ vs $\tilde{z}_i$ para todo $i\in\{1,2,...,n-1\}$.

**6)** Grafique $\hat{z}_{i+1}$ vs $\hat{z}_i$ para todo $i\in\{1,2,...,n-1\}$.

**7)** Comparando los resultados de los incisos **5)** y **6)** con los del gráfico del inciso **3)** del ejercicio **3)**.

Discuta y comente.

In [ ]:
# ==============================================================================
# Ejercicio 4: Señal ruidosa, suavizado con Savitzky-Golay y mapas de Poincaré
# ==============================================================================

# ------------------------------------------------------------------------------
# 4.1) Incorporación de ruido Gaussiano con varianza sigma^2 = 1 (sigma = 1)
# ------------------------------------------------------------------------------
np.random.seed(42)  # Semilla fija para reproducibilidad
sigma = 1.0
ruido = np.random.normal(loc=0.0, scale=sigma, size=len(z_atr))
z_tilde = z_atr + ruido

# Gráfico de la señal ruidosa z_tilde(t) vs t junto con la señal original
plt.figure(figsize=(14, 5))
mask_zoom = (t_atr >= 50.0) & (t_atr <= 90.0)

plt.plot(t_atr[mask_zoom], z_tilde[mask_zoom], color='tab:gray', lw=0.9, alpha=0.8, label=r'$\tilde{z}(t) = z(t) + \mathcal{N}(0, 1)$ (señal con ruido)')
plt.plot(t_atr[mask_zoom], z_atr[mask_zoom], color='tab:green', lw=1.5, label=r'$z(t)$ (señal limpia original)')
plt.title(r'4.1) Curva con ruido $\tilde{z}(t)$ vs $t$ (detalle en $t \in [50, 90]$)', fontsize=13, fontweight='bold')
plt.xlabel('Tiempo $t$', fontsize=11)
plt.ylabel('$z$', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)
plt.show()

# ------------------------------------------------------------------------------
# 4.2) Suavizado con filtro de Savitzky-Golay (window=51, polyorder=3)
# ------------------------------------------------------------------------------
window_length = 51
polyorder = 3
z_hat = sgn.savgol_filter(z_tilde, window_length=window_length, polyorder=polyorder)

print(f"Filtro Savitzky-Golay aplicado: ventana = {window_length} puntos, orden = {polyorder}.")

# ------------------------------------------------------------------------------
# 4.3) Identificación de tiempos \hat{t}_i de máximos en \hat{z}(t) y cálculo de \tilde{z}_i
# ------------------------------------------------------------------------------
# Fijamos distancia mínima de 30 pasos entre picos para evitar falsos máximos residuales
peaks_hat_indices, _ = sgn.find_peaks(z_hat, distance=30)

t_hat = t_atr[peaks_hat_indices]        # Tiempos \hat{t}_i donde ocurren los máximos de \hat{z}
z_hat_peaks = z_hat[peaks_hat_indices]  # Valores de los máximos suavizados \hat{z}_i = \hat{z}(\hat{t}_i)
z_tilde_peaks = z_tilde[peaks_hat_indices]  # Valores de la señal ruidosa \tilde{z}_i = \tilde{z}(\hat{t}_i)

print(f"Número de máximos locales identificados en la curva suavizada: n = {len(t_hat)}")

# ------------------------------------------------------------------------------
# 4.4) Gráfico de \tilde{z}(t) con máximos (\tilde{z}_i, \hat{t}_i) en azul y (\hat{z}_i, \hat{t}_i) en rojo
# ------------------------------------------------------------------------------
plt.figure(figsize=(14, 6))
mask_zoom = (t_atr >= 50.0) & (t_atr <= 90.0)
mask_peaks_zoom = (t_hat >= 50.0) & (t_hat <= 90.0)

plt.plot(t_atr[mask_zoom], z_tilde[mask_zoom], color='lightgray', lw=0.9, label=r'$\tilde{z}(t)$ (serie con ruido)')
plt.plot(t_atr[mask_zoom], z_hat[mask_zoom], color='tab:purple', lw=1.8, label=r'$\hat{z}(t)$ (curva suavizada Savitzky-Golay)')
plt.plot(t_hat[mask_peaks_zoom], z_tilde_peaks[mask_peaks_zoom], 'bo', markersize=6, label=r'Máximos sobre ruidosa $(\hat{t}_i, \tilde{z}_i)$')
plt.plot(t_hat[mask_peaks_zoom], z_hat_peaks[mask_peaks_zoom], 'ro', markersize=6, label=r'Máximos sobre suavizada $(\hat{t}_i, \hat{z}_i)$')

plt.title(r'4.4) Curva suavizada $\hat{z}(t)$ y comparación de máximos $(\hat{t}_i, \tilde{z}_i)$ vs $(\hat{t}_i, \hat{z}_i)$', fontsize=13, fontweight='bold')
plt.xlabel('Tiempo $t$', fontsize=11)
plt.ylabel('$z$', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)
plt.show()

# ------------------------------------------------------------------------------
# 4.5) Mapa de Poincaré con datos ruidosos: \tilde{z}_{i+1} vs \tilde{z}_i
# ------------------------------------------------------------------------------
z_tilde_i = z_tilde_peaks[:-1]
z_tilde_i_plus_1 = z_tilde_peaks[1:]

plt.figure(figsize=(8, 7))
plt.plot(z_tilde_i, z_tilde_i_plus_1, 'o', color='tab:blue', markersize=3.5, alpha=0.7, label=r'$(\tilde{z}_i, \tilde{z}_{i+1})$')
plt.plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.6, label=r'Identidad $z_{i+1} = z_i$')
plt.title(r'4.5) Mapa de Poincaré con datos ruidosos ($\tilde{z}_{i+1}$ vs $\tilde{z}_i$)', fontsize=13, fontweight='bold')
plt.xlabel(r'$\tilde{z}_i$', fontsize=12)
plt.ylabel(r'$\tilde{z}_{i+1}$', fontsize=12)
plt.xlim(z_min, z_max)
plt.ylim(z_min, z_max)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

# ------------------------------------------------------------------------------
# 4.6) Mapa de Poincaré con datos suavizados: \hat{z}_{i+1} vs \hat{z}_i
# ------------------------------------------------------------------------------
z_hat_i = z_hat_peaks[:-1]
z_hat_i_plus_1 = z_hat_peaks[1:]

plt.figure(figsize=(8, 7))
plt.plot(z_hat_i, z_hat_i_plus_1, 'o', color='tab:red', markersize=3.5, alpha=0.7, label=r'$(\hat{z}_i, \hat{z}_{i+1})$')
plt.plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.6, label=r'Identidad $z_{i+1} = z_i$')
plt.title(r'4.6) Mapa de Poincaré con datos suavizados ($\hat{z}_{i+1}$ vs $\hat{z}_i$)', fontsize=13, fontweight='bold')
plt.xlabel(r'$\hat{z}_i$', fontsize=12)
plt.ylabel(r'$\hat{z}_{i+1}$', fontsize=12)
plt.xlim(z_min, z_max)
plt.ylim(z_min, z_max)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# ==============================================================================
# 4.7) Comparación lado a lado de los tres mapas de Poincaré
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

# 1. Mapa original sin ruido (Ejercicio 3.3)
axes[0].plot(z_i, z_i_plus_1, 'o', color='crimson', markersize=3.5, alpha=0.75)
axes[0].plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.5)
axes[0].set_title(r'(a) Original limpio: $z_{i+1}$ vs $z_i$', fontsize=12, fontweight='bold')
axes[0].set_xlabel(r'$z_i$', fontsize=12)
axes[0].set_ylabel(r'$z_{i+1}$', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Mapa con ruido sin filtrar (Ejercicio 4.5)
axes[1].plot(z_tilde_i, z_tilde_i_plus_1, 'o', color='tab:blue', markersize=3.5, alpha=0.6)
axes[1].plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.5)
axes[1].set_title(r'(b) Con ruido: $\tilde{z}_{i+1}$ vs $\tilde{z}_i$', fontsize=12, fontweight='bold')
axes[1].set_xlabel(r'$\tilde{z}_i$', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)

# 3. Mapa filtrado con Savitzky-Golay (Ejercicio 4.6)
axes[2].plot(z_hat_i, z_hat_i_plus_1, 'o', color='tab:red', markersize=3.5, alpha=0.75)
axes[2].plot([z_min, z_max], [z_min, z_max], 'k--', lw=1.2, alpha=0.5)
axes[2].set_title(r'(c) Suavizado Savitzky-Golay: $\hat{z}_{i+1}$ vs $\hat{z}_i$', fontsize=12, fontweight='bold')
axes[2].set_xlabel(r'$\hat{z}_i$', fontsize=12)
axes[2].grid(True, linestyle='--', alpha=0.6)

axes[0].set_xlim(z_min, z_max)
axes[0].set_ylim(z_min, z_max)

plt.suptitle('Comparativa de Secciones de Poincaré: Original vs Ruidoso vs Suavizado', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Discusión:**

### 1. Dinámica del Sistema de Lorenz en función de los parámetros (Ejercicio 2)

El sistema de Lorenz:
$$\frac{dx}{dt} = s(y - x), \quad \frac{dy}{dt} = rx - y - xz, \quad \frac{dz}{dt} = xy - bz$$
modela la convección en una capa bidimensional de fluido calentada térmicamente desde abajo. La dinámica exhibe transiciones cualitativas drásticas a medida que varía el parámetro $r$ (número de Rayleigh normalizado):

1. **Caso i ($s=10, r=0.5, b=8/3$): Régimen de conducción pura ($r < 1$)**
   - El origen $(0,0,0)$ es el único punto de equilibrio del sistema y es **globalmente asintóticamente estable**.
   - Físicamente, representa la conducción térmica en ausencia de convección (el fluido permanece en reposo).
   - Como se evidencia en las series temporales y en el gráfico tridimensional, cualquier condición inicial es atraída monótonamente hacia el origen, disipando toda fluctuación.

2. **Caso ii ($s=10, r=10, b=8/3$): Convección estacionaria ($1 < r < r_H \approx 24.74$)**
   - En $r = 1$, el origen sufre una bifurcación trinchera (*pitchfork bifurcation*) perdiendo estabilidad. Emergen dos nuevos equilibrios simétricos:
     $$C^\pm = \left(\pm\sqrt{b(r-1)}, \, \pm\sqrt{b(r-1)}, \, r-1\right)$$
   - Para $r = 10$, estos puntos corresponden a $C^\pm = (\pm\sqrt{24}, \pm\sqrt{24}, 9) \approx (\pm 4.899, \pm 4.899, 9)$, los cuales representan rodillos convectivos estacionarios girando en sentido horario o antihorario.
   - Las trayectorias describen oscilaciones amortiguadas (foco estable en el subespacio correspondiente) que espiralan hacia uno de los dos puntos fijos atractores.

3. **Caso iii ($s=10, r=28, b=8/3$): Régimen Caótico aperiódico ($r > r_H$)**
   - Para $r > r_H = \frac{s(s+b+3)}{s-b-1} \approx 24.74$, los puntos $C^\pm$ pierden estabilidad mediante una bifurcación de Hopf subcrítica.
   - Dado que el sistema es altamente disipativo ($
     \nabla \cdot f = -(s+1+b) = -41/3 < 0$, contrayendo volúmenes a tasa $e^{-41t/3}$ hacia un conjunto de medida nula), las órbitas quedan confinadas en una región acotada pero no convergen a ningún punto fijo ni órbita periódica estable.
   - El resultado es el célebre **atractor extraño de Lorenz**, caracterizado por movimiento aperiódico con divergencia exponencial de trayectorias próximas (sensibilidad a condiciones iniciales) y alternancia caótica entre los dos lóbulos (alas de la mariposa).

---

### 2. Geometría del Atractor y el Mapa de Retorno de Lorenz (Ejercicio 3)

- En su trabajo pionero de 1963, Edward Lorenz demostró que el flujo continuo en $\mathbb{R}^3$ puede reducirse a un mapa de retorno discreto unidimensional analizando los sucesivos máximos locales de la coordenada $z(t)$:
  $$z_1, z_2, \dots, z_i, z_{i+1}, \dots$$
- La definición de estos máximos corresponde geométricamente a una sección de Poincaré transversal definida por $\dot{z} = 0$ con $\ddot{z} < 0$.
- Debido a la fortísima contracción volumétrica transversal, las trayectorias colapsan sobre una variedad casi bidimensional ramificada. En consecuencia, el par $(z_i, z_{i+1})$ no genera una nube bidimensional dispersa, sino que se alinea de manera casi exacta sobre una curva unidimensional continua con forma de **cúspide** (*tent-like map*):
  $$z_{i+1} \approx M(z_i)$$
- **Significado dinámico fundamental:** La pendiente $|M'(z)|$ de esta función de retorno es estrictamente mayor que 1 en prácticamente todo su dominio ($|M'(z)| > 1$). Esto demuestra analíticamente que:
  1. Cualquier pequeño intervalo de condiciones iniciales $\Delta z$ se estira exponencialmente en cada iteración, garantizando la existencia de un **exponente de Lyapunov positivo** (caos determinista).
  2. No pueden existir puntos periódicos estables (atractores periódicos), ya que la condición de estabilidad $|(M^k)'| < 1$ se viola en todas partes.
  3. El sistema continuo puede modelarse cualitativa y cuantitativamente mediante un mapa unidimensional expansor no inversible.

---

### 3. Efecto del Ruido, Suavizado y Reconstrucción de Secciones de Poincaré (Ejercicio 4)

1. **Observación sobre los máximos temporales (Inciso 4.4):**
   - Los puntos rojos $(\hat{t}_i, \hat{z}_i)$ corresponden a los máximos de la curva suavizada $\hat{z}(t)$, ubicándose sobre la envolvente suave del atractor y muy cercanos a los picos limpios teóricos.
   - Los puntos azules $(\hat{t}_i, \tilde{z}_i)$ son los valores de la serie con ruido evaluada exactamente en los instantes $\hat{t}_i$. Debido a las fluctuaciones aleatorias $\mathcal{N}(0, 1)$, los valores azules oscilan por encima y por debajo de los valores rojos.
   - Resulta fundamental extraer los tiempos de disparo $\hat{t}_i$ a partir de la curva suavizada: si intentáramos encontrar máximos directamente sobre $\tilde{z}(t)$ sin suavizar, las fluctuaciones de alta frecuencia del ruido generarían cientos de máximos locales espurios por ciclo.

2. **Comparación de los Tres Mapas de Poincaré (Incisos 4.5, 4.6 y 4.7):**
   - **Mapa Original Limpio ($z_{i+1}$ vs $z_i$):** Muestra una cúspide unidimensional sumamente definida y nítida. Revela el determinismo subyacente absoluto del sistema dinámico.
   - **Mapa con Ruido Sin Filtrar ($\tilde{z}_{i+1}$ vs $\tilde{z}_i$):** El ruido aditivo blanco $\mathcal{N}(0, 1)$ introduce perturbaciones no correlacionadas en cada instante. Al graficar $\tilde{z}_{i+1} = z_{i+1} + \eta_{i+1}$ versus $\tilde{z}_i = z_i + \eta_i$, las incertidumbres estocásticas ensanchan la curva, transformándola en una nube bidimensional difusa. A partir de una señal experimental con esta dispersión, sería muy difícil distinguir entre un proceso estocástico puro y un atractor caótico determinista contaminado con ruido observacional.
   - **Mapa con Señal Suavizada ($\hat{z}_{i+1}$ vs $\hat{z}_i$):** El filtro de Savitzky-Golay (ventana de 51 puntos y orden 3) ajusta localmente polinomios de grado 3 mediante mínimos cuadrados, filtrando eficientemente el ruido Gaussiano de alta frecuencia mientras preserva con gran fidelidad la curvatura parabólica propia de los máximos de Lorenz. En este mapa de retorno, la nube difusa colapsa nuevamente en una cúspide delgada y bien delimitada, recuperando prácticamente toda la geometría del mapa determinista original.

### Conclusión para Neurociencia y Procesamiento de Señales
En neurofisiología y neurociencia computacional, los registros electrofisiológicos (potenciales de membrana, espigas neuronales, LFP o EEG) combinan dinámicas deterministas complejas y no lineales con ruido de canal, ruido sináptico o ruido de medición. La metodología desarrollada en esta guía demuestra cómo el suavizado no lineal adecuado (como el filtro Savitzky-Golay) junto con el análisis de secciones de Poincaré y mapas de retorno permite recuperar estructuras deterministas de baja dimensión, calcular exponentes de Lyapunov y caracterizar la dinámica intrínseca del sistema aun en presencia de ruido experimental significativo.